# Fine-Tune a UNET Model for MNIST Digit Denoising

This notebook demonstrates how to fine-tune a UNET model for denoising MNIST digits.

## Overview
- Load and preprocess MNIST dataset
- Add noise to create training pairs (noisy → clean)
- Build a UNET architecture
- Train the model to denoise images
- Evaluate and visualize results

## 1. Import Required Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Load and Prepare MNIST Dataset

In [ ]:
# Hyperparameters
BATCH_SIZE = 64
NOISE_LEVEL = 0.3
LEARNING_RATE = 1e-3
NUM_EPOCHS = 10

# Transform to normalize MNIST images
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Load MNIST dataset
train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Training samples: {len(train_dataset)}')
print(f'Test samples: {len(test_dataset)}')

## 3. Create Noise Function

In [ ]:
def add_noise(images, noise_level=0.3):
    """
    Add Gaussian noise to images
    
    Args:
        images: Clean images tensor
        noise_level: Standard deviation of Gaussian noise
    
    Returns:
        Noisy images tensor
    """
    noise = torch.randn_like(images) * noise_level
    noisy_images = images + noise
    noisy_images = torch.clamp(noisy_images, 0., 1.)
    return noisy_images

# Visualize some examples
sample_images, _ = next(iter(test_loader))
sample_noisy = add_noise(sample_images[:8], NOISE_LEVEL)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(sample_images[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[0, i].set_title('Clean')
    
    axes[1, i].imshow(sample_noisy[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')
    axes[1, i].set_title('Noisy')

plt.tight_layout()
plt.show()

## 4. Define UNET Architecture

In [ ]:
class DoubleConv(nn.Module):
    """(Conv -> BatchNorm -> ReLU) * 2"""
    
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """Downscaling with maxpool then double conv"""
    
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    
    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """Upscaling then double conv"""
    
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)
    
    def forward(self, x1, x2):
        x1 = self.up(x1)
        # Concatenate along channel dimension
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class UNet(nn.Module):
    """UNET architecture for image denoising"""
    
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        
        # Encoder (downsampling path)
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        
        # Decoder (upsampling path)
        self.up1 = Up(512, 256)
        self.up2 = Up(256, 128)
        self.up3 = Up(128, 64)
        
        # Output layer
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # Encoder
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        
        # Decoder with skip connections
        x = self.up1(x4, x3)
        x = self.up2(x, x2)
        x = self.up3(x, x1)
        
        # Output
        x = self.outc(x)
        x = self.sigmoid(x)
        return x


# Initialize model
model = UNet(in_channels=1, out_channels=1).to(device)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

## 5. Define Loss Function and Optimizer

In [ ]:
# Loss function: Mean Squared Error for reconstruction
criterion = nn.MSELoss()

# Optimizer: Adam
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True
)

## 6. Training Function

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device, noise_level):
    """
    Train the model for one epoch
    
    Returns:
        Average loss for the epoch
    """
    model.train()
    running_loss = 0.0
    
    progress_bar = tqdm(dataloader, desc='Training')
    for batch_idx, (images, _) in enumerate(progress_bar):
        # Move to device
        clean_images = images.to(device)
        
        # Add noise to create noisy images
        noisy_images = add_noise(clean_images, noise_level)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(noisy_images)
        
        # Compute loss (reconstruct clean images from noisy)
        loss = criterion(outputs, clean_images)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})
    
    return running_loss / len(dataloader)


def validate(model, dataloader, criterion, device, noise_level):
    """
    Validate the model
    
    Returns:
        Average validation loss
    """
    model.eval()
    running_loss = 0.0
    
    with torch.no_grad():
        for images, _ in tqdm(dataloader, desc='Validation'):
            clean_images = images.to(device)
            noisy_images = add_noise(clean_images, noise_level)
            
            outputs = model(noisy_images)
            loss = criterion(outputs, clean_images)
            running_loss += loss.item()
    
    return running_loss / len(dataloader)

## 7. Train the Model

In [ ]:
# Training history
train_losses = []
val_losses = []

print('Starting training...')
for epoch in range(NUM_EPOCHS):
    print(f'\nEpoch {epoch+1}/{NUM_EPOCHS}')
    print('-' * 50)
    
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device, NOISE_LEVEL)
    train_losses.append(train_loss)
    
    # Validate
    val_loss = validate(model, test_loader, criterion, device, NOISE_LEVEL)
    val_losses.append(val_loss)
    
    print(f'Train Loss: {train_loss:.6f}')
    print(f'Val Loss: {val_loss:.6f}')
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Save best model
    if val_loss == min(val_losses):
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
        }, 'best_unet_denoising.pth')
        print('Saved best model!')

print('\nTraining complete!')

## 8. Plot Training History

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Validation Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Training History')
plt.legend()
plt.grid(True)
plt.show()

## 9. Evaluate and Visualize Results

In [ ]:
# Load best model
checkpoint = torch.load('best_unet_denoising.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Get test samples
test_images, test_labels = next(iter(test_loader))
test_images = test_images.to(device)
test_noisy = add_noise(test_images, NOISE_LEVEL)

# Denoise using the model
with torch.no_grad():
    denoised_images = model(test_noisy)

# Move to CPU for visualization
test_images = test_images.cpu()
test_noisy = test_noisy.cpu()
denoised_images = denoised_images.cpu()

# Visualize results
num_samples = 8
fig, axes = plt.subplots(3, num_samples, figsize=(20, 7))

for i in range(num_samples):
    # Original clean image
    axes[0, i].imshow(test_images[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Clean', fontsize=12)
    
    # Noisy image
    axes[1, i].imshow(test_noisy[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Noisy', fontsize=12)
    
    # Denoised image
    axes[2, i].imshow(denoised_images[i].squeeze(), cmap='gray')
    axes[2, i].axis('off')
    if i == 0:
        axes[2, i].set_title('Denoised', fontsize=12)

plt.tight_layout()
plt.savefig('denoising_results.png', dpi=150, bbox_inches='tight')
plt.show()

print('Results saved to denoising_results.png')

## 10. Quantitative Evaluation

In [ ]:
def calculate_psnr(img1, img2):
    """
    Calculate Peak Signal-to-Noise Ratio (PSNR)
    Higher is better
    """
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0:
        return float('inf')
    return 20 * torch.log10(1.0 / torch.sqrt(mse))

# Calculate metrics on test set
model.eval()
psnr_noisy = []
psnr_denoised = []

with torch.no_grad():
    for images, _ in tqdm(test_loader, desc='Calculating metrics'):
        clean_images = images.to(device)
        noisy_images = add_noise(clean_images, NOISE_LEVEL)
        denoised = model(noisy_images)
        
        for i in range(clean_images.size(0)):
            # PSNR for noisy vs clean
            psnr_n = calculate_psnr(clean_images[i], noisy_images[i])
            psnr_noisy.append(psnr_n.item())
            
            # PSNR for denoised vs clean
            psnr_d = calculate_psnr(clean_images[i], denoised[i])
            psnr_denoised.append(psnr_d.item())

print(f'\nAverage PSNR (Noisy vs Clean): {np.mean(psnr_noisy):.2f} dB')
print(f'Average PSNR (Denoised vs Clean): {np.mean(psnr_denoised):.2f} dB')
print(f'Improvement: {np.mean(psnr_denoised) - np.mean(psnr_noisy):.2f} dB')

## 11. Test with Different Noise Levels

In [ ]:
# Test the model's robustness to different noise levels
noise_levels = [0.1, 0.2, 0.3, 0.4, 0.5]
test_image = test_images[0:1].to(device)

fig, axes = plt.subplots(2, len(noise_levels) + 1, figsize=(18, 6))

# Show original
axes[0, 0].imshow(test_image[0].cpu().squeeze(), cmap='gray')
axes[0, 0].axis('off')
axes[0, 0].set_title('Original')
axes[1, 0].axis('off')

# Test different noise levels
with torch.no_grad():
    for idx, noise_lvl in enumerate(noise_levels, 1):
        noisy = add_noise(test_image, noise_lvl)
        denoised = model(noisy)
        
        # Noisy
        axes[0, idx].imshow(noisy[0].cpu().squeeze(), cmap='gray')
        axes[0, idx].axis('off')
        axes[0, idx].set_title(f'Noise={noise_lvl}')
        
        # Denoised
        axes[1, idx].imshow(denoised[0].cpu().squeeze(), cmap='gray')
        axes[1, idx].axis('off')
        axes[1, idx].set_title(f'Denoised')

plt.tight_layout()
plt.show()

## 12. Save Final Model

In [ ]:
# Save the complete model for deployment
torch.save(model, 'unet_denoising_complete.pth')
print('Complete model saved to unet_denoising_complete.pth')

# Save model architecture and weights separately
torch.save({
    'model_state_dict': model.state_dict(),
    'noise_level': NOISE_LEVEL,
    'architecture': 'UNet',
    'input_channels': 1,
    'output_channels': 1,
}, 'unet_denoising_weights.pth')
print('Model weights saved to unet_denoising_weights.pth')

## Summary

This notebook demonstrated:

1. **Data Preparation**: Loaded MNIST and added Gaussian noise to create training pairs
2. **Model Architecture**: Built a UNET with encoder-decoder structure and skip connections
3. **Training**: Fine-tuned the model to denoise images using MSE loss
4. **Evaluation**: Visualized results and calculated PSNR metrics
5. **Robustness Testing**: Tested the model with various noise levels

### Next Steps:
- Experiment with different noise types (salt & pepper, Poisson)
- Try deeper UNET architectures
- Test on other datasets (Fashion-MNIST, CIFAR-10)
- Implement residual connections or attention mechanisms
- Use perceptual loss functions for better quality